# 02 – Baseline tomato vs non_tomato model (small sanity run)

This notebook wires together:

- `data/splits/chips_index.parquet` built in **01_build_splits_and_check.ipynb**,
- `AlphaEarthChipsDataset` (`src/modeling/alpha_earth_dataset.py`),
- `AlphaEarthTomatoModel` (`src/modeling/alpha_earth_model.py`).

It runs a **tiny training loop** on a small subset of the data to make sure
the modeling stack works end-to-end (no full training yet). The real training
on all chips will run later on SageMaker GPU using the same pieces.


In [ ]:
from pathlib import Path, sys

repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root))
print("Repo root added to sys.path:", repo_root)


In [ ]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader, Subset
import pandas as pd

from src.utils.paths import REPO_ROOT, load_paths_config
from src.modeling.alpha_earth_dataset import AlphaEarthChipsDataset, collate_chips
from src.modeling.alpha_earth_model import AlphaEarthTomatoModel

cfg = load_paths_config()
splits_dir = REPO_ROOT / cfg.get("data", {}).get("splits", "data/splits")
index_parquet = splits_dir / "chips_index.parquet"
index_csv = splits_dir / "chips_index.csv"

print("Repo root:", REPO_ROOT)
print("Splits dir:", splits_dir)
print("Index parquet exists:", index_parquet.is_file())
print("Index csv exists:", index_csv.is_file())


## 1. Create small train/val datasets

We take a small subset (e.g. 256 chips per split) to keep this notebook
fast on CPU. Full training will be on SageMaker GPU using the same code.


In [ ]:
index_path = index_parquet if index_parquet.is_file() else index_csv
print("Using index:", index_path)

train_ds_full = AlphaEarthChipsDataset(index_path=index_path, split="train")
val_ds_full = AlphaEarthChipsDataset(index_path=index_path, split="val")

print("Full train size:", len(train_ds_full))
print("Full val size:", len(val_ds_full))

subset_n = 256
train_indices = list(range(min(subset_n, len(train_ds_full))))
val_indices = list(range(min(subset_n, len(val_ds_full))))

train_ds = Subset(train_ds_full, train_indices)
val_ds = Subset(val_ds_full, val_indices)

train_loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_chips,
)
val_loader = DataLoader(
    val_ds,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_chips,
)

batch = next(iter(train_loader))
print("Batch image shape:", batch.image.shape)
print("Batch valid_mask shape:", batch.valid_mask.shape)
print("Batch label shape:", batch.label.shape)


## 2. Define model and simple loss

We use a very small encoder + two heads:
- pixel logits (for per-pixel tomato probability),
- chip logits (for chip-level tomato/non_tomato).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AlphaEarthTomatoModel(in_channels=64, base_channels=32, dropout_p=0.1).to(device)

bce_logits = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

lambda_chip = 1.0
lambda_pixel = 0.2


## 3. Tiny training loop (1–2 epochs)

This is only to verify that the model and dataset work; we keep it short.


In [ ]:
def step_batch(batch, train: bool = True):
    model.train(train)
    x = batch.image.to(device)
    mask = batch.valid_mask.to(device)
    y = batch.label.to(device).float()  # (B,)

    pixel_logits, chip_logits = model(x)
    # Chip loss: BCE on chip logits vs label.
    chip_loss = bce_logits(chip_logits, y)

    # Pixel loss: weak supervision, use same label for all valid pixels in chip.
    # Broadcast labels to (B,1,H,W).
    y_px = y.view(-1, 1, 1, 1).expand_as(pixel_logits)
    pixel_loss_raw = bce_logits(pixel_logits, y_px)
    # Mask out invalid pixels by scaling loss.
    # Approximation: scale by fraction of valid pixels.
    valid_frac = mask.float().mean().clamp(min=1e-3)
    pixel_loss = pixel_loss_raw * valid_frac

    total = lambda_chip * chip_loss + lambda_pixel * pixel_loss
    return total, chip_loss.detach().cpu(), pixel_loss.detach().cpu()


num_epochs = 1
for epoch in range(1, num_epochs + 1):
    running = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        loss, chip_l, pix_l = step_batch(batch, train=True)
        loss.backward()
        optimizer.step()
        running += loss.item()
    avg_loss = running / max(1, len(train_loader))
    print(f"Epoch {epoch}: train loss = {avg_loss:.4f}")


This is **only** a baseline sanity check notebook. For full training on GPU,
we will reuse the same `AlphaEarthChipsDataset` and `AlphaEarthTomatoModel`
inside a SageMaker notebook, with more epochs and proper logging to
`logs/experiments/...`.
